# **04. Experiments and Final Training Pipeline**

This notebook runs the reusable training pipeline from `src/customer_support_ai/train.py`. It compares candidate models on the validation split, selects the best model by macro F1-score, retrains it using training plus validation data, and evaluates it on the held-out test split.

# **1. Environment Setup**

We locate the project root and use the notebook kernel's Python executable. This avoids environment mismatch between VS Code, Jupyter, and the project `.venv`.

In [1]:
# Import standard libraries for running the training pipeline.
from pathlib import Path
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src' / 'customer_support_ai').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from customer_support_ai.config import COMPATIBLE_DATASET_FILENAMES, COMPATIBLE_DATASET_PATHS

RESULTS_PATH = PROJECT_ROOT / 'results' / 'metrics_summary.csv'

print(f'Project root: {PROJECT_ROOT}')
print('Compatible CSV files:')
for filename in COMPATIBLE_DATASET_FILENAMES:
    print(f'- {filename}')
print(f'Python executable: {sys.executable}')

Project root: /Users/mariorubiano/Library/CloudStorage/OneDrive-Personal/Documents/UTS/Semester 4/AI/Assignments/AT3/trn-uts-aipa
Compatible CSV files:
- aa_dataset-tickets-multi-lang-5-2-50-version.csv
- dataset-tickets-multi-lang-4-20k.csv
- dataset-tickets-multi-lang3-4k.csv
Python executable: /Users/mariorubiano/Library/CloudStorage/OneDrive-Personal/Documents/UTS/Semester 4/AI/Assignments/AT3/trn-uts-aipa/.venv/bin/python


# **2. Run Training Pipeline**

The training command uses the three compatible Kaggle CSVs by default. The pipeline saves model files, metrics, classification reports, confusion matrices, and the processed dataset.

In [2]:
# Run the implementation training script on the compatible merged dataset.
missing_files = [path.name for path in COMPATIBLE_DATASET_PATHS if not path.exists()]
if missing_files:
    raise FileNotFoundError('Place these files in data/raw/: ' + ', '.join(missing_files))

completed = subprocess.run(
    [sys.executable, '-m', 'customer_support_ai.train'],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

print(completed.stdout)

    task               model      split  accuracy  macro_precision  macro_recall  macro_f1  weighted_f1
category          linear_svm       test  0.530714         0.507242      0.547319  0.524646     0.530600
category          linear_svm validation  0.515507         0.477816      0.513708  0.493164     0.516123
category logistic_regression validation  0.429088         0.381896      0.475032  0.408271     0.434263
category         naive_bayes validation  0.391749         0.570440      0.209067  0.201916     0.331944
priority          linear_svm       test  0.580849         0.566755      0.569027  0.567824     0.581097
priority          linear_svm validation  0.567600         0.547937      0.549990  0.548877     0.568009
priority logistic_regression validation  0.540349         0.526236      0.536869  0.527821     0.543624
priority         naive_bayes validation  0.494279         0.492498      0.419667  0.389717     0.451140



# **3. Model Comparison Results**

Validation rows compare candidate models. Test rows show the final selected model performance after retraining on training plus validation data.

In [3]:
# Load the saved metrics table produced by the pipeline.
metrics = pd.read_csv(RESULTS_PATH)
metrics.sort_values(['task', 'split', 'macro_f1'], ascending=[True, True, False])

,task,model,split,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,category,linear_svm,test,0.530714,0.507242,0.547319,0.524646,0.530600
1,category,linear_svm,validation,0.515507,0.477816,0.513708,0.493164,0.516123
2,category,logistic_regression,validation,0.429088,0.381896,0.475032,0.408271,0.434263
3,category,naive_bayes,validation,0.391749,0.570440,0.209067,0.201916,0.331944
4,priority,linear_svm,test,0.580849,0.566755,0.569027,0.567824,0.581097
5,priority,linear_svm,validation,0.567600,0.547937,0.549990,0.548877,0.568009
6,priority,logistic_regression,validation,0.540349,0.526236,0.536869,0.527821,0.543624
7,priority,naive_bayes,validation,0.494279,0.492498,0.419667,0.389717,0.451140


# **4. Selected Models**

For each task, the selected model is the one with the highest validation macro F1-score.

In [4]:
# Identify the selected validation winner and final test result for each task.
validation_results = metrics[metrics['split'] == 'validation']
test_results = metrics[metrics['split'] == 'test']

selected_models = (
    validation_results
    .sort_values(['task', 'macro_f1'], ascending=[True, False])
    .groupby('task')
    .head(1)
    .merge(test_results[['task', 'model', 'macro_f1', 'accuracy']], on=['task', 'model'], how='left', suffixes=('_validation', '_test'))
)

selected_models

,task,model,split,accuracy_validation,macro_precision,macro_recall,macro_f1_validation,weighted_f1,macro_f1_test,accuracy_test
0,category,linear_svm,validation,0.515507,0.477816,0.513708,0.493164,0.516123,0.524646,0.530714
1,priority,linear_svm,validation,0.567600,0.547937,0.549990,0.548877,0.568009,0.567824,0.580849


# **5. Saved Artifacts**

These files provide evidence for the report and presentation. The raw CSV remains local and is not committed to GitHub.

In [5]:
# List generated outputs from training.
artifact_paths = [
    PROJECT_ROOT / 'data' / 'processed' / 'model_ready_tickets.csv',
    PROJECT_ROOT / 'models' / 'category_model.pkl',
    PROJECT_ROOT / 'models' / 'priority_model.pkl',
    PROJECT_ROOT / 'results' / 'dataset_profile.json',
    PROJECT_ROOT / 'results' / 'metrics_summary.csv',
    PROJECT_ROOT / 'results' / 'classification_reports.json',
    PROJECT_ROOT / 'results' / 'confusion_matrix_category.png',
    PROJECT_ROOT / 'results' / 'confusion_matrix_priority.png',
]

artifacts = pd.DataFrame({
    'artifact': [str(path.relative_to(PROJECT_ROOT)) for path in artifact_paths],
    'exists': [path.exists() for path in artifact_paths],
    'size_kb': [round(path.stat().st_size / 1024, 1) if path.exists() else None for path in artifact_paths],
})

artifacts

,artifact,exists,size_kb
0,data/processed/model_ready_tickets.csv,True,93938.4
1,models/category_model.pkl,True,3602.1
2,models/priority_model.pkl,True,1961.4
3,results/dataset_profile.json,True,0.6
4,results/metrics_summary.csv,True,1.1
5,results/classification_reports.json,True,11.6
6,results/confusion_matrix_category.png,True,171.4
7,results/confusion_matrix_priority.png,True,53.3


# **6. Experiment Interpretation**

The results should be interpreted carefully. The pipeline is reproducible, but performance is limited by weak relationships between the available features and the target labels.

In [6]:
# Summarise the experimental findings for the report.
for _, row in selected_models.iterrows():
    print(
        f"- {row['task'].title()} selected model: {row['model']} | "
        f"validation macro F1 = {row['macro_f1_validation']:.4f} | "
        f"test macro F1 = {row['macro_f1_test']:.4f}"
    )

print('- Results are clearly above the dummy baseline from notebook 03, supporting the dataset replacement decision.')
print('- The system is still decision support, so high-priority or uncertain cases should remain human-reviewed.')

- Category selected model: linear_svm | validation macro F1 = 0.4932 | test macro F1 = 0.5246
- Priority selected model: linear_svm | validation macro F1 = 0.5489 | test macro F1 = 0.5678
- Results are clearly above the dummy baseline from notebook 03, supporting the dataset replacement decision.
- The system is still decision support, so high-priority or uncertain cases should remain human-reviewed.
